# Custom Data Augmenters

In this tutorial, we will guide you through the process of building custom data augmenters and integrating them into the data flow. Specifically, we will cover how to:

1. Create and configure a custom augmenter to expand or filter datasets.
2. Implement a data augmenter that works with both individual samples and batches.
3. Apply the custom augmenter to a dataset, enabling efficient data manipulation within the pipeline.

## Imports

In this section, we will import all the necessary libraries and components that we need for creating and applying a custom data augmenter.

In [1]:
import datasets
from typing import Iterable
from hyped import DataFlow
from hyped.typing import Int, TraceIndexList
from hyped.core import BaseDataAugmenter, BaseDataAugmenterConfig, RunContext, process_mode

## Creating the Dataset and Data Flow instance

In this section, we will create a sample dataset and instantiate the DataFlow object that will manage the data transformations.

In [2]:
# Create a sample dataset
ds = datasets.Dataset.from_dict({"x": [0, 1, 2, 3, 4]})

# Create a data flow instance
flow = DataFlow(ds.features)

## Implementing the Custom Augmenter

Here we will define a custom augmenter, which will expand the dataset by generating new samples based on a configurable parameter `n`. This is achieved by subclassing `BaseDataAugmenter`.

In [3]:
# Define configuration for the custom augmenter
class CustomAugmenterConfig(BaseDataAugmenterConfig):
    n: int  # Number of times to replicate the input value

# Implement the custom augmenter
class CustomAugmenter(BaseDataAugmenter[CustomAugmenterConfig]):

    def process(self, ctx: RunContext, x: Int) -> Iterable[Int]:
        # Yield the value `x` repeated `n` times
        yield from (x for _ in range(self.config.n))

`CustomAugmenterConfig` extends `BaseDataAugmenterConfig` to specify the configuration for the augmenter, which includes an integer `n` that defines how many times to replicate each value.

`CustomAugmenter` is the main augmenter class, extending `BaseDataAugmenter`. The process method takes in an input value `x` and yields it `n` times, creating new variations of the original dataset.

## Applying the Custom Augmenter

In this section, we will apply the custom augmenter to the dataset within the data flow graph and observe the results.

In [4]:
# Apply the custom augmenter
out = CustomAugmenter(n=3).call(x=flow.source["x"])

# Apply the augmentation to the dataset and collect the results
out_ds = flow.apply(ds, collect={"out": out})

# Display the augmented dataset
print(out_ds.to_dict())

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

{'out': [0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4]}


The `CustomAugmenter(n=3)` creates an instance of the custom augmenter, where `n=3` specifies that each value in the dataset will be repeated `3` times. We call the augmenter with the data (`x=flow.source["x"]`), and the results are applied to the dataset using `flow.apply()`.

## Batched Processing

In this section, we will extend the custom augmenter to handle batched processing. Batched processing is useful when you want to apply augmentations over multiple samples at once, increasing efficiency, especially with large datasets.

In [ ]:
# Define configuration for the custom augmenter
class CustomAugmenterConfigV2(BaseDataAugmenterConfig):
    n: int  # Number of times to replicate the input value

# Implement the custom augmenter
class CustomAugmenterV2(BaseDataAugmenter[CustomAugmenterConfigV2]):
    @process_mode(batched=True)
    def process(self, ctx: RunContext, x: Int) -> tuple[Int, TraceIndexList]:
        # Yield the value `x` repeated `n` times
        return x * self.config.n, list(range(len(ctx.index))) * self.config.n

We define a new configuration class `CustomAugmenterConfigV2`, similar to the previous one, with an added integer `n` to specify how many times the input should be repeated.

The process method now uses the `@process_mode(batched=True)` decorator, indicating that this method handles batched processing. This decorator ensures that the method is capable of handling multiple samples in one call.

The method returns two outputs:
- The first output is the augmented data, where each input value `x` is repeated `n` times.
- The second output is a list of indices, which indicates the origin of each output sample. This list is required to track where each sample came from in the input data.

In [5]:
# Apply the custom augmenter
out = CustomAugmenterV2(n=3).call(x=flow.source["x"])

# Apply the augmentation to the dataset and collect the results
out_ds = flow.apply(ds, collect={"out": out})

# Display the augmented dataset
print(out_ds.to_dict())

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

{'out': [0, 1, 2, 3, 4, 0, 1, 2, 3, 4, 0, 1, 2, 3, 4]}


We apply the custom augmenter exactly the same as before. However, note that the output dataset is essentially the same as before but rearranges. 